# 6.12 CatBoost

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.12_catboost.ipynb)

Companion code for Section 6.12 of the book. Run the cells in order from top to bottom.

**Example 6.12.1** · bash

```bash
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score, classification_report
import numpy as np
import pandas as pd

# ── On numerical data ─────────────────────────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      stratify=y, random_state=42)

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    eval_metric='AUC',
    early_stopping_rounds=50,
    random_seed=42,
    verbose=0
)
cat_model.fit(X_train, y_train, eval_set=(X_test, y_test))

print(f'CatBoost Test Accuracy: {cat_model.score(X_test, y_test):.4f}')
print(f'CatBoost Test AUC:      {roc_auc_score(y_test, cat_model.predict_proba(X_test)[:,1]):.4f}')
print(f'Best iteration:         {cat_model.best_iteration_}')

# ── Native categorical handling ────────────────────────────────────
# CatBoost handles categoricals directly - no encoding needed!
df_cat = pd.DataFrame({
    'age':       np.random.randint(18, 70, 500),
    'income':    np.random.exponential(50000, 500),
    'education': np.random.choice(['HS','BS','MS','PhD'], 500),
    'city':      np.random.choice(['London','NY','Paris','Tokyo','Berlin'], 500),
    'gender':    np.random.choice(['Male','Female'], 500),
    'target':    np.random.choice([0,1], 500, p=[0.7,0.3])
})

X_cat = df_cat.drop('target', axis=1)
y_cat = df_cat['target']
cat_features = ['education','city','gender']   # Tell CatBoost which are categorical

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_cat, y_cat, test_size=0.2, stratify=y_cat, random_state=42)

cat_native = CatBoostClassifier(iterations=200, learning_rate=0.1,
                                  depth=4, random_seed=42, verbose=0)
cat_native.fit(X_tr_c, y_tr_c, cat_features=cat_features)
print(f'\nCatBoost with native categoricals: {cat_native.score(X_te_c, y_te_c):.4f}')

# ── Three-way comparison ──────────────────────────────────────────
import xgboost as xgb
import lightgbm as lgb

print('\n=== XGBoost vs LightGBM vs CatBoost - Final Comparison ===')
print(f'{'Model':<15} {'Test Acc':>10} {'Test AUC':>10} {'Notes'}')
print('='*65)

xgb_m = xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                            eval_metric='logloss', use_label_encoder=False,
                            random_state=42, verbosity=0)
xgb_m.fit(X_train, y_train)
print(f'{'XGBoost':<15} {xgb_m.score(X_test,y_test):>10.4f}
{roc_auc_score(y_test,xgb_m.predict_proba(X_test)[:,1]):>10.4f}  Second-order gradients, regularisation')

lgb_m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05,
                             num_leaves=31, random_state=42, verbose=-1)
lgb_m.fit(X_train, y_train)
print(f'{'LightGBM':<15} {lgb_m.score(X_test,y_test):>10.4f}
{roc_auc_score(y_test,lgb_m.predict_proba(X_test)[:,1]):>10.4f}  Fastest; leaf-wise growth; large datasets')

print(f'{'CatBoost':<15} {cat_model.score(X_test,y_test):>10.4f}
{roc_auc_score(y_test,cat_model.predict_proba(X_test)[:,1]):>10.4f}  Best for categoricals; ordered boosting')
```